# Notebook 3 - Time-Based Train / Validation / Test Split

## Decision

We use a chronological 70% / 15% / 15% split. A late-delivery model will be trained on past orders and used on future orders, so a time split is a more realistic estimate than a random split. It also prevents information from future periods influencing earlier decisions.

In [1]:
from pathlib import Path
import json
import warnings
warnings.filterwarnings("ignore")

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
ARTIFACTS = ROOT / "artifacts"
FIGURES = ROOT / "figures"
REPORTS = ROOT / "reports"
for folder in [ARTIFACTS, FIGURES, REPORTS]:
    folder.mkdir(parents=True, exist_ok=True)
print(f"Task root: {ROOT.resolve()}")

Task root: C:\Users\moath\Desktop\MLOPS\MLOps_Task1_Rand_Salem\MLOps-Qafza-2026\Tasks\Task-02


In [2]:
import pandas as pd

source = ARTIFACTS / "02_labeled_orders.parquet"
assert source.exists(), "Run Notebook 2 first."
data = pd.read_parquet(source)
data["order_purchase_timestamp"] = pd.to_datetime(data["order_purchase_timestamp"], errors="raise")
data = data.sort_values(["order_purchase_timestamp", "order_id"]).reset_index(drop=True)
print("Date range:", data["order_purchase_timestamp"].min(), "to", data["order_purchase_timestamp"].max())

Date range: 2016-09-15 12:16:38 to 2018-08-29 15:00:37


## 1. Create chronological splits

In [3]:
n = len(data)
train_end = int(n * 0.70)
validation_end = int(n * 0.85)

train = data.iloc[:train_end].copy()
validation = data.iloc[train_end:validation_end].copy()
test = data.iloc[validation_end:].copy()

assert len(train) + len(validation) + len(test) == len(data)
assert train["order_id"].is_unique and validation["order_id"].is_unique and test["order_id"].is_unique
assert set(train["order_id"]).isdisjoint(validation["order_id"])
assert set(train["order_id"]).isdisjoint(test["order_id"])
assert set(validation["order_id"]).isdisjoint(test["order_id"])
assert train["order_purchase_timestamp"].max() <= validation["order_purchase_timestamp"].min()
assert validation["order_purchase_timestamp"].max() <= test["order_purchase_timestamp"].min()

## 2. Check date range and label balance

In [4]:
split_frames = {"train": train, "validation": validation, "test": test}
split_summary = pd.DataFrame([
    {
        "split": name,
        "rows": len(frame),
        "percentage": 100 * len(frame) / len(data),
        "start_date": frame["order_purchase_timestamp"].min(),
        "end_date": frame["order_purchase_timestamp"].max(),
        "late_orders": int(frame["is_late"].sum()),
        "late_rate": frame["is_late"].mean(),
    }
    for name, frame in split_frames.items()
])
split_summary

,split,rows,percentage,start_date,end_date,late_orders,late_rate
0,train,67529,70.000000,2016-09-15 12:16:38,2018-04-15 20:12:35,6096,0.090272
1,validation,14470,14.999482,2018-04-15 20:17:11,2018-06-21 08:29:29,773,0.053421
2,test,14471,15.000518,2018-06-21 08:41:07,2018-08-29 15:00:37,957,0.066132


In [5]:
if split_summary["late_rate"].max() - split_summary["late_rate"].min() > 0.05:
    print("Warning: label prevalence shifted materially over time; keep this as a realistic production challenge.")
else:
    print("Label prevalence is reasonably stable across chronological splits.")

Label prevalence is reasonably stable across chronological splits.


## 3. Save split artifacts

In [6]:
paths = {}
for name, frame in split_frames.items():
    path = ARTIFACTS / f"03_{name}.parquet"
    frame.to_parquet(path, index=False)
    paths[name] = path.name

summary_records = split_summary.assign(
    start_date=split_summary["start_date"].astype(str),
    end_date=split_summary["end_date"].astype(str),
).to_dict(orient="records")
(REPORTS / "03_split_summary.json").write_text(json.dumps(summary_records, indent=2), encoding="utf-8")
print("Saved:", paths)

Saved: {'train': '03_train.parquet', 'validation': '03_validation.parquet', 'test': '03_test.parquet'}
